In [8]:
import os
import librosa
import resampy
import numpy as np
from sklearn.model_selection import train_test_split

In [9]:
def prepare_dataset(data_path):
    X = []
    y = []
    
    # Define our classes
    classes = ['focus', 'distracted']
    
    for label, category in enumerate(classes):
        folder_path = os.path.join(data_path, category)
        print(f"Processing {category}...")
        
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.wav'):
                try:
                    # 1. Load audio (ESC-50 files are 5 seconds long)
                    file_path = os.path.join(folder_path, file_name)
                    audio, sr = librosa.load(file_path, duration=5, res_type='kaiser_fast')
                    
                    # 2. Extract Mel Spectrogram
                    # n_mels=128 gives us a "height" of 128 pixels
                    spectrogram = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
                    
                    # 3. Convert to Log Scale (Decibels)
                    log_spec = librosa.power_to_db(spectrogram, ref=np.max)
                    
                    # 4. Standardize Shape
                    # Audio clips might vary slightly in length. We need exactly 128x216 pixels.
                    # (216 is the standard width for a 5s clip at default sampling)
                    if log_spec.shape[1] > 216:
                        log_spec = log_spec[:, :216]
                    else:
                        pad_width = 216 - log_spec.shape[1]
                        log_spec = np.pad(log_spec, ((0, 0), (0, pad_width)), mode='constant')
                    
                    X.append(log_spec)
                    y.append(label)
                except Exception as e:
                    print(f"Error processing {file_name}: {e}")

    # Convert lists to NumPy arrays
    X = np.array(X)
    y = np.array(y)
    
    # 5. Normalization (Very important for Neural Networks!)
    # Scales values from (-80 to 0) to (0 to 1)
    X = (X - np.min(X)) / (np.max(X) - np.min(X))
    
    # 6. Reshape for CNN
    # CNNs expect (Height, Width, Channels). Since it's grayscale, Channels = 1.
    X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)
    
    return X, y

# EXECUTION
X, y = prepare_dataset('D:/ML/zen_identifier/data')

# Split into Training and Testing sets (80% to learn, 20% to test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Successfully loaded {len(X)} samples.")
print(f"Training shape: {X_train.shape}") # Should be (N, 128, 216, 1)

Processing focus...
Processing distracted...
Successfully loaded 240 samples.
Training shape: (192, 128, 216, 1)


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [11]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 216, 1)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(2, activation='softmax')
])

d:\ML\zen_identifier\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [13]:
history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_test, y_test),
                    batch_size=32)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 231ms/step - accuracy: 0.6354 - loss: 1.0837 - val_accuracy: 0.5000 - val_loss: 0.6841
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 194ms/step - accuracy: 0.6302 - loss: 0.6603 - val_accuracy: 0.6875 - val_loss: 0.6208
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - accuracy: 0.6823 - loss: 0.6011 - val_accuracy: 0.6875 - val_loss: 0.5325
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step - accuracy: 0.8021 - loss: 0.4439 - val_accuracy: 0.6875 - val_loss: 0.4593
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 182ms/step - accuracy: 0.8490 - loss: 0.3884 - val_accuracy: 0.8958 - val_loss: 0.3182
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 195ms/step - accuracy: 0.8438 - loss: 0.3767 - val_accuracy: 0.8333 - val_loss: 0.3214
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 193ms/step - accuracy: 0.9219 - loss: 0.2607 - val_accuracy: 0.8750 - val_loss: 0.2650
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 187ms/step - accuracy: 0.9062 - loss: 0.2485 - val_accuracy: 0.8958 - val_loss:

In [14]:
model.save('zen_focus_model.h5')
print("Model trained and saved as zen_focus_model.h5")

Model trained and saved as zen_focus_model.h5
